# Lección 7.5 — Pipelines: evitar fugas mientras se ajusta

> **Módulo 7 · Regularización y ajuste de hiperparámetros** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.3 (preparación de datos), Lección 2.6 (fuga de datos), Lección 4.6 (validación cruzada), Lección 7.3 (búsqueda de hiperparámetros)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué **cualquier paso que aprende de los datos** puede filtrar información durante la validación cruzada.
- Construir un `Pipeline` (y un `ColumnTransformer` para datos mixtos) y ajustarlo con `GridSearchCV` / `RandomizedSearchCV`.
- Nombrar hiperparámetros dentro de un pipeline con la sintaxis `paso__parametro`.
- Detectar fugas por **grupos** y estimar honestamente un procedimiento de ajuste con **validación cruzada anidada**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import (train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV,
                                     KFold, GroupKFold)

warnings.filterwarnings("ignore")
print("✅ Todo listo")

---
## 1. 📖 La intuición: la guía de estudio que copia el examen

Un profesor quiere saber si su método de enseñanza funciona. Prepara una **guía de estudio**, los alumnos estudian con ella y luego hacen el examen. Sacan 9.5 de promedio. ¡Éxito!

Pero hay un detalle: el profesor preparó la guía **mirando el examen**. Sin querer, eligió justo los temas que iban a salir. Ese 9.5 **no mide** lo que los alumnos saben: mide lo bien que la guía "adivinó" el examen.

En ML, la **validación cruzada** es el examen. Y la "guía de estudio" es todo lo que hacemos **antes** de entrenar: escalar, rellenar faltantes, elegir columnas… Si esos pasos se calculan con **todos** los datos, incluidos los pliegues de validación, el modelo "estudió con el examen a la vista". Esto es **fuga de datos** (*data leakage*, Lección 2.6), y durante el ajuste de hiperparámetros es especialmente traicionera porque **repites** el examen cientos de veces.

La solución es una **receta cerrada**: un **pipeline**, que aplica todos los pasos **dentro** de cada pliegue, aprendiendo solo del entrenamiento de ese pliegue.

### ✋ Ejercicio 1 — ¿Aprende o no aprende?
Un paso "aprende de los datos" si necesita mirarlos para saber qué hacer (tiene un `.fit`). ¿Cuáles de estos pasos aprenden de los datos y, por tanto, pueden causar fuga si se calculan antes de la validación cruzada?
1. Escalar con la media y la desviación de cada columna (`StandardScaler`).
2. Convertir metros a kilómetros (dividir entre 1,000).
3. Rellenar faltantes con la mediana de la columna.
4. Quedarte con las 20 columnas más relacionadas con la etiqueta.
5. Borrar la columna `id_cliente`.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Aprenden: 1, 3 y 4.** Necesitan calcular algo con los datos (media, mediana, relación con la etiqueta). Si lo calculan con todos los datos, los pliegues de validación "se cuelan" en el cálculo.

**No aprenden: 2 y 5.** Dividir entre 1,000 o borrar una columna es igual para cualquier fila, sin mirar nada. Se pueden hacer antes sin problema.

El más peligroso es el **4**, porque mira **la etiqueta**. Ahora verás por qué.
</details>

---
## 2. 📖 El concepto: una fuga que fabrica resultados de la nada

Vamos a hacer el experimento más famoso sobre fugas (aparece en el libro clásico *The Elements of Statistical Learning*).

- **200 pacientes** y **5,000 columnas** de números aleatorios (piensa en mediciones genéticas).
- Una etiqueta **al azar** (moneda al aire): no hay **nada** que aprender. Ningún modelo honesto puede superar el 50 %.

El plan: elegir las 20 columnas más relacionadas con la etiqueta (`SelectKBest`) y entrenar una regresión logística. Lo haremos de dos maneras:
- ❌ **Con fuga:** primero elegimos las 20 columnas **con todos los datos**, y después hacemos validación cruzada.
- ✅ **Con pipeline:** la selección va **dentro** de la validación cruzada.

### 🔮 Ejercicio 2 — Predice
¿Qué exactitud esperas en cada caso?

✍️ *Mis predicciones:*

In [ ]:
rng = np.random.default_rng(0)
X_genes = rng.normal(size=(200, 5_000))
y_genes = rng.integers(0, 2, 200)              # ¡pura moneda al aire!

# ❌ Con fuga: la selección mira TODOS los datos (también los que luego servirán de validación)
X_elegidas = SelectKBest(f_classif, k=20).fit_transform(X_genes, y_genes)
con_fuga = cross_val_score(LogisticRegression(), X_elegidas, y_genes, cv=5).mean()

# ✅ Con pipeline: en cada pliegue, la selección solo mira el entrenamiento de ese pliegue
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
sin_fuga = cross_val_score(pipe, X_genes, y_genes, cv=5).mean()

print(f"❌ Seleccionar antes de validar : exactitud CV = {con_fuga:.1%}")
print(f"✅ Seleccionar dentro del pipeline: exactitud CV = {sin_fuga:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- ❌ Con fuga: ≈ **79 %**. ¡Con datos que son puro ruido!
- ✅ Con pipeline: ≈ **51 %**. Lo que corresponde a una moneda al aire.

¿Qué pasó? Entre 5,000 columnas al azar, algunas se parecen a la etiqueta **por casualidad**, y `SelectKBest` las encuentra… **mirando también las filas de validación**. Luego esas columnas "predicen" bien la validación porque fueron elegidas por parecerse a ella. Es la guía de estudio que copia el examen.

Con el pipeline, en cada pliegue la selección solo ve el entrenamiento; las columnas elegidas por casualidad no tienen por qué funcionar con las filas de validación, y el resultado vuelve a la realidad.
</details>

---
## 3. 📖 Cómo funciona un pipeline por dentro

Un `Pipeline` es una lista de pasos con nombre. Todos menos el último son **transformadores** (escalan, rellenan, seleccionan…); el último es el **modelo**.

```
                 fit(X_train, y_train)                         predict(X_nuevo)
 ┌───────────────────────────────────────────┐   ┌──────────────────────────────────────┐
 │ paso 1: imputar  → aprende medianas       │   │ paso 1: imputar  → usa esas medianas │
 │ paso 2: escalar  → aprende medias/desv.   │   │ paso 2: escalar  → usa esas medias   │
 │ paso 3: modelo   → aprende coeficientes   │   │ paso 3: modelo   → predice           │
 └───────────────────────────────────────────┘   └──────────────────────────────────────┘
```

- Con `fit`, cada paso **aprende** con los datos de entrenamiento y le pasa los datos transformados al siguiente.
- Con `predict` o `score`, cada paso **solo aplica** lo que aprendió. Nunca vuelve a aprender con los datos nuevos.
- Dentro de `cross_val_score` o `GridSearchCV`, **todo el pipeline** se reentrena en cada pliegue. Por eso no hay fuga.

Hay dos formas de crearlo:
- `make_pipeline(StandardScaler(), LogisticRegression())` → pone los nombres solo (`standardscaler`, `logisticregression`).
- `Pipeline([("escalar", StandardScaler()), ("modelo", LogisticRegression())])` → tú eliges los nombres.

In [ ]:
pipe = Pipeline([
    ("seleccion", SelectKBest(f_classif, k=20)),
    ("modelo",    LogisticRegression()),
])
print(pipe)
print("\nPasos:", list(pipe.named_steps))

### 📖 Ajustar hiperparámetros **dentro** del pipeline
Para buscar hiperparámetros de un paso, se escribe **`nombre_del_paso__parametro`** (con **dos** guiones bajos). Así puedes ajustar a la vez el modelo **y** el preprocesamiento: por ejemplo, cuántas columnas seleccionar.

### ✋ Ejercicio 3 — Escribe los nombres
Con el pipeline de arriba, ¿cómo se llaman en la rejilla…
1. …el `k` de la selección de columnas?
2. …el `C` de la regresión logística?

Y si lo hubieras creado con `make_pipeline(SelectKBest(...), LogisticRegression())`?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. `"seleccion__k"`
2. `"modelo__C"`

Con `make_pipeline`, los nombres son el nombre de la clase en minúsculas: `"selectkbest__k"` y `"logisticregression__C"`.

> 💡 Si dudas, `pipe.get_params().keys()` te muestra todos los nombres válidos.
</details>

In [ ]:
rejilla = {"seleccion__k": [5, 20, 100], "modelo__C": [0.01, 1, 100]}
busqueda = GridSearchCV(pipe, rejilla, cv=5).fit(X_genes, y_genes)
print("Mejores:", busqueda.best_params_)
print(f"Mejor exactitud CV: {busqueda.best_score_:.1%}")

### 🔮 Ejercicio 4 — ¿Y ahora qué número sale?
La búsqueda de arriba usa el pipeline (sin fuga) y se queda con la mejor de 9 combinaciones. Antes de mirar la salida: ¿esperas un `best_score_` cercano al 50 %, al 79 % o algo intermedio? ¿Por qué?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

Algo **un poco por encima del 50 %** (sale ≈ 55.5 %). No hay fuga, pero sí el **optimismo por selección** de la Lección 7.3: es el mejor de 9 intentos, y el mejor de varios intentos siempre sale algo inflado por la suerte. Ahora verás cómo medir eso de forma honesta (Sección 6).
</details>

---
## 4. 📖 Datos reales: columnas numéricas y categóricas con faltantes

En la vida real las tablas mezclan números y categorías, y tienen huecos. `ColumnTransformer` aplica una receta distinta a cada tipo de columna, y todo va dentro del pipeline.

Datos: 1,500 solicitudes de un seguro de auto. Queremos predecir si el cliente **tendrá un siniestro** este año.

In [ ]:
rng = np.random.default_rng(7)
n = 1_500
seguros = pd.DataFrame({
    "edad":            rng.integers(18, 80, n).astype(float),
    "antiguedad_auto": rng.integers(0, 20, n).astype(float),
    "km_anuales":      rng.normal(15_000, 6_000, n).clip(1_000).round(),
    "multas_previas":  rng.poisson(0.6, n).astype(float),
    "tipo_auto":       rng.choice(["sedán", "SUV", "deportivo", "pickup"], n, p=[.45, .3, .1, .15]),
    "zona":            rng.choice(["urbana", "suburbana", "rural"], n),
})
riesgo = (-2.2 + 1.0 * seguros.multas_previas + 0.12 * (30 - seguros.edad).clip(0)       # conductores jóvenes
          + 0.00008 * (seguros.km_anuales - 15_000) + 1.5 * (seguros.tipo_auto == "deportivo")
          + 0.8 * (seguros.zona == "urbana"))
seguros["siniestro"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)

# Huecos al azar en algunas columnas, como en cualquier sistema real
for col, pct in [("edad", .08), ("km_anuales", .12), ("tipo_auto", .05)]:
    seguros.loc[rng.random(n) < pct, col] = np.nan

X_seg, y_seg = seguros.drop(columns="siniestro"), seguros["siniestro"]
X_seg_tr, X_seg_te, y_seg_tr, y_seg_te = train_test_split(X_seg, y_seg, test_size=0.25,
                                                          stratify=y_seg, random_state=0)
print(f"Siniestros: {y_seg.mean():.0%}")
print("Faltantes por columna:", X_seg.isna().sum()[lambda s: s > 0].to_dict())
X_seg.head()

Ahora el pipeline completo. Léelo de adentro hacia afuera:
- **Numéricas:** rellenar con la mediana → escalar.
- **Categóricas:** rellenar con la más frecuente → *one-hot* (Lección 2.3).
- **Modelo:** regresión logística.

In [ ]:
numericas = Pipeline([("imputar", SimpleImputer(strategy="median")),
                      ("escalar", StandardScaler())])
categoricas = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                        ("onehot",  OneHotEncoder(handle_unknown="ignore"))])

preprocesador = ColumnTransformer([
    ("num", numericas,   make_column_selector(dtype_include="number")),
    ("cat", categoricas, make_column_selector(dtype_include=object)),
])

modelo_seguros = Pipeline([("prep", preprocesador), ("modelo", LogisticRegression(max_iter=1000))])
cv = cross_val_score(modelo_seguros, X_seg_tr, y_seg_tr, cv=5, scoring="roc_auc")
print(f"AUC en validación cruzada: {cv.mean():.3f} ± {cv.std():.3f}")
modelo_seguros

### ✏️ Ejercicio 5 — Ajusta el preprocesamiento y el modelo a la vez
El nombre de un parámetro "profundo" encadena los nombres de los pasos: por ejemplo, la estrategia de imputación numérica es `prep__num__imputar__strategy`.

Ejecuta la búsqueda. Luego ✏️ añade a la rejilla `"prep__cat__onehot__min_frequency": [None, 50]` (agrupa las categorías raras) y vuelve a ejecutar.

In [ ]:
rejilla = {
    "prep__num__imputar__strategy": ["mean", "median"],
    "modelo__C": [0.01, 0.1, 1, 10],
    # ✏️ añade aquí: "prep__cat__onehot__min_frequency": [None, 50],
}
busqueda_seg = GridSearchCV(modelo_seguros, rejilla, cv=5, scoring="roc_auc").fit(X_seg_tr, y_seg_tr)
print("Mejores:", busqueda_seg.best_params_)
print(f"Mejor AUC CV: {busqueda_seg.best_score_:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La mejor combinación usa `C = 10`, aunque `C = 1` y `C = 10` empatan en la práctica (AUC ≈ **0.729**); con `C = 0.01` baja un poco (≈ 0.719). La estrategia de imputación da igual: media y mediana difieren en el cuarto decimal. Al añadir `min_frequency`, el resultado no cambia: aquí no hay categorías raras que agrupar (la menos frecuente, "deportivo", aparece en ~10 % de las filas, más de 50 veces).

Lo importante es la **forma**: en una sola búsqueda decidiste cosas del preprocesamiento y del modelo, y **ninguna decisión miró los pliegues de validación**. Y conectando con la Lección 7.4: la imputación resultó ser una perilla de las que casi no importan; `C` es la crítica.
</details>

---
## 5. 📖 ¿Qué pasa si…? No todas las fugas son igual de graves

### 🔮 Ejercicio 6 — El escalado fuera del pipeline
Es el error más común: escalar **todos** los datos antes de la validación cruzada. ¿Crees que inflará el resultado tanto como la selección de columnas (de 51 % a 79 %)?

✍️ *Mi predicción:*

In [ ]:
X_num = X_seg_tr.select_dtypes("number").fillna(X_seg_tr.median(numeric_only=True))

X_escalado_todo = StandardScaler().fit_transform(X_num)       # ❌ escala mirando todas las filas
con_fuga = cross_val_score(LogisticRegression(), X_escalado_todo, y_seg_tr, cv=5, scoring="roc_auc").mean()
sin_fuga = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X_num, y_seg_tr,
                           cv=5, scoring="roc_auc").mean()
print(f"❌ Escalado fuera: AUC {con_fuga:.4f}")
print(f"✅ Escalado dentro: AUC {sin_fuga:.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No**: la diferencia es minúscula (en el cuarto decimal). La media y la desviación de una columna casi no cambian por incluir unas filas más, y además **no miran la etiqueta**.

Entonces, ¿da igual? **No**, por tres razones:
1. Con pocos datos, valores extremos o pasos más "codiciosos" (selección de columnas, *target encoding*, sobremuestreo con SMOTE), la fuga crece mucho, como viste en la Sección 2.
2. En producción, el modelo **tiene** que llevar consigo el escalado aprendido en el entrenamiento: el pipeline lo empaqueta todo en un solo objeto.
3. Es un hábito: si siempre usas pipelines, nunca tienes que preguntarte "¿esta fuga será grave?".

| Paso | ¿Mira la etiqueta? | Gravedad típica de la fuga |
|---|---|---|
| Escalar, imputar | No | 🟡 Baja |
| PCA y similares | No | 🟡 Baja–media |
| Selección de columnas por relación con `y` | **Sí** | 🔴 Alta |
| *Target encoding* | **Sí** | 🔴 Alta |
| Sobremuestreo (SMOTE) antes de dividir | Copia filas entre pliegues | 🔴 Alta |
</details>

### 🔮 Ejercicio 7 — Fuga por grupos: el mismo paciente en dos pliegues
Un hospital tiene **60 pacientes** con **10 mediciones** cada uno (600 filas). Las mediciones de un mismo paciente se parecen mucho entre sí. La etiqueta es por paciente y, para el experimento, **la ponemos al azar**: no hay nada que aprender.

Usamos un KNN con `k=1` (Lección 5.3) y validación cruzada:
- `KFold` normal: reparte **filas** al azar.
- `GroupKFold`: reparte **pacientes**; las 10 filas de un paciente van juntas.

¿Qué exactitud esperas con cada uno?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(1)
n_pacientes, n_mediciones = 60, 10
perfil = rng.normal(size=(n_pacientes, 5))                 # cada paciente tiene su "huella"
etiqueta = rng.integers(0, 2, n_pacientes)                 # al azar: no hay señal real
X_pac = np.repeat(perfil, n_mediciones, axis=0) + rng.normal(0, 0.1, (n_pacientes * n_mediciones, 5))
y_pac = np.repeat(etiqueta, n_mediciones)
paciente = np.repeat(np.arange(n_pacientes), n_mediciones)

knn = KNeighborsClassifier(n_neighbors=1)
print(f"KFold (reparte filas)        : {cross_val_score(knn, X_pac, y_pac, cv=KFold(5, shuffle=True, random_state=0)).mean():.0%}")
print(f"GroupKFold (reparte pacientes): {cross_val_score(knn, X_pac, y_pac, cv=GroupKFold(5), groups=paciente).mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `KFold`: **100 %**. Cada fila de validación tiene a su "gemela" (otra medición del mismo paciente) en el entrenamiento, y KNN simplemente copia su etiqueta. El modelo no aprendió medicina: **reconoce pacientes**.
- `GroupKFold`: ≈ **48 %**. Ante pacientes **nuevos**, el modelo es una moneda al aire, que es la verdad.

Ningún pipeline arregla esto: la fuga está en **cómo partes los datos**. Pregúntate siempre: *"¿Con qué datos se usará el modelo en la vida real?"*. Si serán **pacientes nuevos**, la validación debe tener pacientes nuevos. Lo mismo con clientes, tiendas o máquinas repetidos, y con el **tiempo**: para predecir el futuro, valida con el futuro (`TimeSeriesSplit`, Lección 4.6). En `GridSearchCV`, pasa el divisor con `cv=GroupKFold(5)` y los grupos con `.fit(X, y, groups=paciente)`.
</details>

---
## 6. 📖 Validación cruzada anidada: evaluar el procedimiento completo

En la Lección 7.3 viste que `best_score_` es optimista: es el mejor de muchos intentos. Si tienes pocos datos y no quieres "gastar" un conjunto de prueba, la solución es la **validación cruzada anidada**:

```
Bucle EXTERNO (evaluar):  divide los datos en 5 pliegues
   └── en cada pliegue externo:
         Bucle INTERNO (ajustar): GridSearchCV con su propia validación cruzada, SOLO con la parte de entrenamiento
         → el ganador se evalúa en el pliegue externo, que la búsqueda nunca vio
Resultado: el promedio de los 5 pliegues externos
```

Mide **el procedimiento completo** ("buscar hiperparámetros con este método y luego entrenar"), no un modelo concreto. En código es sorprendentemente corto: un `GridSearchCV` dentro de un `cross_val_score`.

### 🔮 Ejercicio 8 — ¿Qué número dará?
Volvemos a los 5,000 genes al azar. La búsqueda de la Sección 3 dio un `best_score_` de ≈ 55.5 %. ¿Cuánto crees que dará la validación cruzada anidada?

✍️ *Mi predicción:*

In [ ]:
busqueda_interna = GridSearchCV(pipe, {"seleccion__k": [5, 20, 100], "modelo__C": [0.01, 1, 100]}, cv=5)
anidada = cross_val_score(busqueda_interna, X_genes, y_genes, cv=5)        # ¡una búsqueda dentro de cada pliegue!
print(f"best_score_ de la búsqueda   : {busqueda.best_score_:.1%}   (optimista)")
print(f"Validación cruzada anidada   : {anidada.mean():.1%}   (honesta)")

<details><summary><b>👀 Ver respuesta</b></summary>

La anidada da ≈ **49.5 %**: el nivel del azar. Confirma que **el procedimiento entero** no aprende nada de estos datos, aunque la búsqueda sola prometía ≈ 55.5 %.

⚠️ La anidada **no te da un modelo**: te da una **estimación honesta**. Para el modelo final, ejecutas la búsqueda una vez con todos los datos de entrenamiento. Cuesta mucho cómputo (aquí: 5 × 9 × 5 = 225 entrenamientos), así que úsala sobre todo con datos pequeños, cuando no puedes permitirte un conjunto de prueba aparte.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Resultados "demasiado buenos" en validación cruzada que no se repiten en producción | Preprocesamiento (sobre todo si mira `y`) calculado con todos los datos | Meter **todos** los pasos que aprenden en un `Pipeline` |
| `scaler.fit(X_test)` o `fit_transform` sobre la prueba | Aprender de la prueba | En la prueba, **solo** `transform` / `predict` (el pipeline lo hace solo) |
| Exactitud altísima con datos que tienen varias filas por persona/tienda/máquina | Fuga por **grupos** | `GroupKFold` y `groups=` |
| Validación excelente y producción mala en datos con fechas | Fuga **temporal** (entrenar con el futuro) | `TimeSeriesSplit` |
| SMOTE / sobremuestreo y luego validación cruzada | Copias de la misma fila en entrenamiento y validación | Sobremuestrear **dentro** del pipeline (p. ej. con `imblearn.pipeline`) |
| `best_score_` reportado como desempeño final | Optimismo por selección | Conjunto de prueba aparte o validación cruzada anidada |
| Error `Invalid parameter` en la búsqueda | Nombre mal escrito (un solo guion bajo, paso mal nombrado) | `pipe.get_params().keys()` |

### 💾 Guardar y usar el pipeline final
Una vez elegido, el pipeline completo (preprocesamiento + modelo) se guarda como **un solo objeto**:

```python
import joblib
joblib.dump(busqueda_seg.best_estimator_, "modelo_seguros.joblib")   # guarda TODO: imputación, escalado, one-hot y modelo
modelo = joblib.load("modelo_seguros.joblib")
modelo.predict_proba(nuevas_solicitudes)                             # tabla "cruda", con huecos y texto: el pipeline se encarga
```
Esto evita otro error clásico: preprocesar distinto en producción que en el entrenamiento (Lección 15.1).

---
## 8. 📋 Ficha de la herramienta

| | `Pipeline` (+ `ColumnTransformer`) |
|---|---|
| **Qué es** | Una receta cerrada: pasos de preparación + modelo, que se entrenan y aplican juntos |
| **Intuición** | La guía de estudio se prepara solo con el temario, nunca con el examen |
| **Cómo evita fugas** | En cada pliegue, todos los pasos aprenden solo del entrenamiento de ese pliegue |
| **Sintaxis clave** | `make_pipeline(...)`, `Pipeline([("nombre", paso), ...])`, `ColumnTransformer`, `paso__parametro` |
| **✅ Ventajas** | Sin fugas · Ajusta preprocesamiento y modelo a la vez · Un solo objeto para producción |
| **❌ Lo que NO arregla** | Fugas por grupos o por tiempo (eso depende de cómo divides) · Columnas que contienen la respuesta (Lección 2.6) |

## 9. 🧠 Resumen en 5 puntos
1. Todo paso que **aprende de los datos** (escalar, imputar, seleccionar, codificar, sobremuestrear) debe ir **dentro** de un pipeline para no filtrar información de la validación.
2. Las fugas más graves son las que **miran la etiqueta** (selección de columnas, *target encoding*): pueden fabricar buenos resultados con datos que son puro ruido.
3. Dentro de un pipeline, los hiperparámetros se nombran `paso__parametro`, y puedes ajustar a la vez el preprocesamiento y el modelo.
4. Si hay **grupos** (varias filas por paciente) o **tiempo**, la forma de dividir debe imitar el uso real: `GroupKFold`, `TimeSeriesSplit`.
5. La **validación cruzada anidada** estima honestamente el procedimiento completo de ajuste; `best_score_` no.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál de estos pasos **no** necesita ir dentro del pipeline?
- a) `StandardScaler` · b) `SimpleImputer(strategy="median")` · c) Borrar la columna `id` · d) `SelectKBest`

**2.** En `Pipeline([("prep", ...), ("svm", SVC())])`, el hiperparámetro `C` de la SVM se escribe:
- a) `"C"` · b) `"svm_C"` · c) `"svm__C"` · d) `"SVC__C"`

**3.** Tienes 5 radiografías por paciente. ¿Qué divisor usas en la validación cruzada?
- a) `KFold` · b) `StratifiedKFold` · c) `GroupKFold` · d) Da igual

**4.** La validación cruzada anidada sirve para…
- a) Entrenar más rápido · b) Estimar honestamente el desempeño del procedimiento de ajuste · c) Elegir el mejor `k` de KNN · d) Evitar los faltantes

**5.** ¿Cuál de estas fugas es, normalmente, la **más grave**?
- a) Escalar con todos los datos · b) Imputar con la mediana de todos los datos · c) Elegir columnas por su correlación con `y` usando todos los datos · d) Convertir metros a km

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Borrar una columna no aprende nada de los datos.
2. **c)** Nombre del paso + **dos** guiones bajos + parámetro.
3. **c)** Las radiografías de un paciente deben ir todas al mismo pliegue.
4. **b)** Mide el procedimiento completo, con la búsqueda incluida.
5. **c)** Mira la etiqueta: puede fabricar señal donde no la hay (Sección 2).
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"Yo no uso pipelines: primero hago `train_test_split`, luego `scaler.fit(X_train)` y aplico `transform` a la prueba. Así no hay fuga"*. ¿Tiene razón? ¿Qué problema aparece si después usa `GridSearchCV` sobre `X_train` ya escalado?

**B. Caso de análisis 💼** Una cadena de supermercados predice ventas diarias de **50 tiendas** durante **3 años**. El equipo usó `KFold` con filas mezcladas y obtuvo un error excelente, pero en producción el modelo falla. Menciona **dos** fugas posibles y cómo validarías correctamente.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Tiene razón **a medias**. Para la evaluación final en la prueba, su método no tiene fuga. Pero dentro de `GridSearchCV`, el escalado ya se aprendió con **todo** `X_train`, incluidos los pliegues de validación internos: hay una fuga (pequeña, porque el escalado no mira `y`) en la **elección** de hiperparámetros. Si en vez de escalar hiciera selección de columnas, la fuga sería grave. Con un pipeline no tiene que pensar en nada de esto.

**B.** (1) **Fuga temporal**: al mezclar días, el modelo entrena con el futuro (p. ej. con datos de diciembre de 2024 para predecir noviembre de 2024). (2) **Fuga por grupos** si el modelo se usará en **tiendas nuevas**: cada tienda aparece en entrenamiento y validación. Validación correcta: `TimeSeriesSplit` (entrenar con el pasado, validar con el período siguiente) y, si el objetivo incluye tiendas nuevas, dejar tiendas completas fuera (`GroupKFold`). Además, cualquier variable como "ventas promedio de la tienda" debe calcularse solo con el pasado.
</details>

### 10.3 🧩 Práctica integradora: la auditoría del notebook "milagroso"

Un compañero presume de un modelo que detecta **clientes que abandonarán** un banco con un **AUC de casi 0.89**. Te pasa su código para que lo revises antes de presentarlo a dirección. Los datos tienen 400 clientes, 300 columnas (muchas, irrelevantes), y algunos faltantes.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos**

In [ ]:
from sklearn.datasets import make_classification
X_b, y_b = make_classification(n_samples=400, n_features=300, n_informative=4, n_redundant=0,
                               flip_y=0.15, random_state=3)
X_b = pd.DataFrame(X_b, columns=[f"v{i}" for i in range(300)])
X_b = X_b.mask(np.random.default_rng(3).random(X_b.shape) < 0.05)       # 5 % de huecos
print(X_b.shape, "· faltantes:", int(X_b.isna().sum().sum()))

**Paso 2 ✋ — Lee el código del compañero y encuentra las fugas.** No lo ejecutes todavía: léelo y anota **cada línea problemática**. Hay **tres** problemas.

In [ ]:
# ===== Código del compañero (¡con errores!) =====
X_imp = SimpleImputer(strategy="mean").fit_transform(X_b)                       # (1)
X_sel = SelectKBest(f_classif, k=10).fit_transform(X_imp, y_b)                  # (2)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_sel, y_b, test_size=0.25, random_state=0)

mejor_auc, mejor_C = 0, None
for C in [0.001, 0.01, 0.1, 1, 10, 100]:                                        # (3)
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C)).fit(X_tr_c, y_tr_c)
    auc = cross_val_score(m, X_te_c, y_te_c, cv=5, scoring="roc_auc").mean()
    if auc > mejor_auc:
        mejor_auc, mejor_C = auc, C
print(f"AUC reportado por el compañero: {mejor_auc:.3f} (C={mejor_C})")

✍️ *Las fugas que encontré:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Línea (1):** imputa con la media de **todas** las filas, incluidas las de prueba. Fuga leve, pero fuga.
2. **Línea (2):** elige las 10 columnas **mirando la etiqueta de todas las filas**, incluida la prueba. ¡Fuga grave! (Sección 2).
3. **Bloque (3):** elige `C` **mirando la prueba** (evalúa cada `C` con `X_te_c`) y además reporta el máximo de 6 intentos sobre esa misma prueba. La prueba se usó para decidir: ya no es un examen honesto.

(Bonus: la validación cruzada sobre solo 100 filas de prueba es además muy ruidosa).
</details>

**Paso 3 ▶️ — Rehazlo bien.** Un solo pipeline con imputación, selección, escalado y modelo; la búsqueda se hace solo con el entrenamiento.

In [ ]:
X_b_tr, X_b_te, y_b_tr, y_b_te = train_test_split(X_b, y_b, test_size=0.25, stratify=y_b, random_state=0)

pipe_b = Pipeline([
    ("imputar",   SimpleImputer(strategy="median")),
    ("seleccion", SelectKBest(f_classif)),
    ("escalar",   StandardScaler()),
    ("modelo",    LogisticRegression(max_iter=1000)),
])
rejilla_b = {"seleccion__k": [4, 10, 30, 100], "modelo__C": [0.01, 0.1, 1, 10]}
busqueda_b = GridSearchCV(pipe_b, rejilla_b, cv=5, scoring="roc_auc").fit(X_b_tr, y_b_tr)
print("Mejores:", busqueda_b.best_params_)
print(f"Mejor AUC CV (algo optimista): {busqueda_b.best_score_:.3f}")

**Paso 4 🔮 — El número honesto.** Antes de ejecutar: ¿el AUC en la prueba se parecerá más al ≈ 0.885 del compañero o al `best_score_` de tu búsqueda?

✍️ *Mi apuesta:*

In [ ]:
from sklearn.metrics import roc_auc_score
auc_prueba = roc_auc_score(y_b_te, busqueda_b.predict_proba(X_b_te)[:, 1])
print(f"AUC del compañero (con fugas)     : {mejor_auc:.3f}")
print(f"Tu best_score_ (CV, sin fugas)    : {busqueda_b.best_score_:.3f}")
print(f"Tu AUC en prueba (una sola vez)   : {auc_prueba:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El AUC honesto en prueba (≈ **0.81**) se parece a tu `best_score_` (≈ 0.81) y está **claramente por debajo** del ≈ 0.885 del compañero. El modelo **sí** aprende algo (hay 4 columnas útiles de verdad), pero mucho menos de lo prometido. La diferencia la fabricaron las fugas, sobre todo elegir columnas mirando las etiquetas de todas las filas.

Fíjate también en el `k` elegido: **4** columnas, justo el número de columnas útiles que tienen los datos. La búsqueda sin fugas descubrió que casi todo lo demás es ruido.
</details>

**Paso 5 ✋ — El mensaje para dirección.** Escribe en 2–3 frases, sin jerga, qué le dirías a dirección sobre el 0.89.

✍️ *Mi mensaje:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

*"El primer resultado (89 %) estaba inflado por un error de método: el modelo se evaluó con información que, en la vida real, no tendría al momento de predecir. Corregido el método, el modelo distingue a los clientes que se van con una calidad de alrededor del 81 %: sigue siendo útil para priorizar llamadas de retención, pero hay que planificar con esa cifra, no con la primera."*
</details>

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, *Common pitfalls and recommended practices* (fugas de datos) y *Pipelines and composite estimators*.
- 📖 **Libro:** Hastie, Tibshirani y Friedman, *The Elements of Statistical Learning*, sección 7.10.2: *"The wrong and right way to do cross-validation"* (el experimento de la Sección 2).
- 📄 **Artículo:** Kapoor y Narayanan (2023), *Leakage and the reproducibility crisis in ML-based science*: cuántos estudios publicados cayeron en estas trampas.

---
⬅️ **Anterior:** [7.4 Cómo priorizar qué hiperparámetros ajustar primero](7.4-priorizar-hiperparametros.ipynb) · ➡️ **Siguiente:** [7.6 Interpretabilidad: importancia de variables, SHAP y dependencias parciales](7.6-interpretabilidad.ipynb)